In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tol_colors as tc
import matplotlib.animation as animation
from pathlib import Path


import tools
import my_plot as mplt

from plot_utils import set_size, set_colors, width_jcap, sci_formatter
plt.style.use('my_style.mplstyle')

In [2]:
def extract_state_scattering_basis(folder_path, ell, ell_max, k, basis="scattering"):
    """Extract scattering basis transfer function data from files in a folder """
    folder = Path(folder_path)
    file_path = []
    file_path2 = []
    # Collect multipole files (0,1,2,...)
    for i in range(ell+1):
        file_path.append(next(folder.glob(f"full_state_kv_{k}.dat_ymax15_lmax{ell_max}_adiabatic_scattering-basis_{i}.dat")))
        # theta values always come from the computational basis for the scattering
        file_path2.append(next(folder.glob(f"full_state_kv_{k}.dat_ymax15_lmax{ell_max}_adiabatic_computation-basis_{i}.dat"))) 
    # Collect background file
    bg_file = next(folder.glob(f"full_state_kv_{k}.dat_ymax15_lmax{ell_max}_adiabatic_scattering-basis_bg.dat"))
    bg_file2 = next(folder.glob(f"full_state_kv_{k}.dat_ymax15_lmax{ell_max}_adiabatic.dat"))

    dtheta = [[] for _ in range(ell+1)]  # To store corrections to the monopole, will be filled from bg_file2
    theta = [[] for _ in range(ell+2)]
    mu = [[] for _ in range(ell+2)]
    y = [[] for _ in range(ell+2)]
    a = []

    # Read header to find theta_corr_0 column (format: "name [col_num]")
    with open(bg_file2, 'r') as f:
        header_line = f.readline().strip('# \n')
        header = header_line.split()
    # Find column index for theta_corr_0: look for "theta_corr_0", then get number from next element "[N]"
    idx = header.index("theta_corr_0")
    theta_corr_col = int(header[idx + 1].strip('[]')) - 1  # Convert to 0-indexed

    # First: background
    data_bg = np.loadtxt(bg_file, comments='#')
    data_theta_bg = np.loadtxt(bg_file2, comments='#')
    
    a.extend(data_bg[:, 0])  # First column is redshift (only read once)
    theta[0].extend(data_theta_bg[:, 1])
    y[0].extend(data_bg[:, 2])
    mu[0].extend(data_bg[:, 9])

    # Then: multipoles
    for i in range(ell+1):
        data = np.loadtxt(file_path[i], comments='#')
        data_theta = np.loadtxt(file_path2[i], comments='#')
        theta[i+1].extend(data_theta[:, 1])
        y[i+1].extend(data[:, 2])
        mu[i+1].extend(data[:, 9])
        dtheta[i].extend(data_theta_bg[:, theta_corr_col+i])
    
    # Load corrections to the monopole from the background file (dynamically found column)
    

    return a, np.array(theta), np.array(y), np.array(mu), np.array(dtheta)

In [ ]:
ell = 1
kv_values = [ "002", "003", "004"]
kv_values_Mpc = {"001" : 1, "002" : 0.1, "003" : 0.01, "004" : 0.001}

zcon = [5.0e6,5.0e5, 5e4, 5e3]

In [ ]:
theta = []
y = []
mu = []
a = []
dtheta = []

for kv in kv_values:
    zt, th, yy, m, dth = extract_state_scattering_basis("../Data/Dark-photons/outputs_dp_ell/outputs-dp1.0_ph/batch/FH/yn_T_aniso_system", ell, 150, kv )
    #zt, th, yy, m, dth = extract_state_scattering_basis("../Data/Dark-photons/output_dp_tfph2/outputs-dp1.0_ph/batch/FH/yn_T_aniso_system", ell, 150, kv )
    #zt, th, yy, m, dth = extract_state_scattering_basis("../Data/Dark-photons/dp-aniso1.0/", ell, 50, kv )
        
    theta.append(th)
    y.append(yy)
    mu.append(m)
    a.append(zt)    
    dtheta.append(dth)

In [3]:
Z_MIN  = 5e3
Z_MAX  = 1e6
N_FRAMES = 120
z_conv_grid = np.logspace(np.log10(Z_MIN), np.log10(Z_MAX), N_FRAMES)


In [ ]:
def make_animation(x, tf_list, output_file="transfer_function.mp4"):
 
    fig, ax = plt.subplots(figsize=(8, 5), facecolor=BG)
    ax.set_facecolor(BG)
 
    ax.set_xscale("log")
    ax.set_xlabel(r"$\nu$ [GHz]", fontsize=13)          # cambia se è k o altro
    ax.set_ylabel(r"Transfer function $T$", fontsize=13)
 
    # limiti fissi sull'asse y per non "saltare" tra frame
    all_vals = np.concatenate(tf_list)
    ymin, ymax = all_vals.min(), all_vals.max()
    margin = 0.05 * (ymax - ymin)
    ax.set_ylim(ymin - margin, ymax + margin)
    ax.set_xlim(x.min(), x.max())
 
    line, = ax.plot([], [], color=VIOLET, lw=2)
    title = ax.set_title("", fontsize=13)
 
    # linea verticale opzionale per z_res se noto
    # ax.axvline(x=NU_RES, color=GREEN, ls="--", alpha=0.5, label=r"$\nu_\mathrm{res}$")
 
    ax.legend(loc="upper right", fontsize=11)
    fig.tight_layout()
 
    def init():
        line.set_data([], [])
        return (line,)
 
    def update(frame):
        z   = z_conv_grid[frame]
        tf  = tf_list[frame]
        line.set_data(x, tf)
        title.set_text(rf"$z_\mathrm{{conv}} = {z:.2e}$")
        return line, title
 
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=N_FRAMES,
        init_func=init,
        interval=1000 / 15,   # ms per frame → ~15 fps; aumenta per rallentare
        blit=True,
    )
 
    # ── esporta ──────────────────────────────────────────────────────────────
    writer = animation.FFMpegWriter(fps=15, bitrate=1800)
    ani.save(output_file, writer=writer, dpi=150)
    print(f"Salvato: {output_file}")
    plt.close(fig)
 
 



In [ ]:


DATA_DIR = "./cosmo_outputs"   # <── cambia con il tuo path HPC/locale

x, tf_list = load_data(DATA_DIR)
make_animation(x, tf_list, output_file="TF_animation.mp4")